In [1]:
from pathlib import Path
from typing import cast

from dataeval.config import set_seed
from datasets import Dataset
from datasets import load_dataset as hf_load

# Seed the random state used by the BoVW vocabulary (MiniBatchKMeans) and the
# KMeans clusterer so the sweep produces identical numbers across machines and
# CI runs. Without this, results vary because random_state defaults to None.
set_seed(42)

# Load MNIST train split
mnist_train = cast(Dataset, hf_load("ylecun/mnist", split="train"))

# Save a subset of 1000 images to disk for the tutorial
# This size is large enough to contain natural similarities between digits
data_path = Path("./data/mnist_sweep")
mnist_train.select(range(1000)).save_to_disk(str(data_path))

/home/aweng/2033/dataeval-flow/.nox/docs/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saving the dataset (0/1 shards):   0%|                                                                                                                                     | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 197928.55 examples/s]

Saving the dataset (1/1 shards): 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 177830.24 examples/s]

In [2]:
from dataeval_flow.config import (
    BoVWExtractorConfig,
    HuggingFaceDatasetConfig,
    ParameterSweepTaskConfig,
    ParameterSweepWorkflowConfig,
    PipelineConfig,
    SourceConfig,
)
from dataeval_flow.workflow import run_task

# Define the sweep workflow
sweep_workflow = ParameterSweepWorkflowConfig(
    name="mnist_sensitivity_sweep",
    # Outlier parameters (sweeping threshold)
    outlier_method=["adaptive"],
    outlier_threshold=[2.5, 3.5, 4.5],
    outlier_flags=["dimension", "pixel", "visual"],
    # Duplicate parameters (sweeping sensitivity)
    # High sensitivity (3.0) flags digits that are visually similar but not identical
    # Low sensitivity (0.5) only flags digits that are extremely similar
    duplicate_cluster_sensitivity=[0.5, 2.0, 3.0],
    duplicate_cluster_algorithm=["hdbscan"],
    duplicate_merge_near=True,
)

# Define the task referencing the sweep workflow
task = ParameterSweepTaskConfig(
    name="mnist_param_sweep", workflow="mnist_sensitivity_sweep", sources="mnist_src", extractor="bovw_ext"
)

# Build the full pipeline config
config = PipelineConfig(
    datasets=[
        HuggingFaceDatasetConfig(name="mnist_sweep", path=str(data_path)),
    ],
    sources=[
        SourceConfig(name="mnist_src", dataset="mnist_sweep"),
    ],
    extractors=[
        BoVWExtractorConfig(name="bovw_ext", vocab_size=256, batch_size=32),
    ],
    workflows=[sweep_workflow],
    tasks=[task],
)

In [3]:
result = run_task(task, config, cache_dir=Path("./cache"))

In [4]:
print(result.report())


  PARAMETER SWEEP COMPLETE. 9 COMBINATIONS EVALUATED.
  Timestamp:    2026-06-09T17:46:17.168135+00:00
  Duration:     0.29s
  Source:       mnist_src (mnist_sweep)
  Model:        bovw_ext (bovw)
--------------------------------------------------------------------------------

  SUMMARY
  -------
  Outliers Sweep ................................. 3 unique combinations  [..]
  Near Duplicates Sweep .......................... 3 unique combinations  [..]

  Health: All checks passed [ok]

  OUTLIERS SWEEP                                           3 unique combinations
  Effect of outlier_threshold on outliers.

  outlier_threshold  Outliers
  -----------------  --------
  2.5                       7
  3.5                       2
  4.5                       1

  NEAR DUPLICATES SWEEP                                    3 unique combinations
  Effect of duplicate_cluster_sensitivity on near duplicates.

  duplicate_cluster_sensitivity  Near Duplicates
  -----------------------------  -----